In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv('.env')
client = OpenAI(
#    api_key=os.getenv('OPENAI_API_KEY')
)

In [19]:
# 위도와 경도를 입력하면 섭씨온도를 return 함수
import requests
def get_weather(latitude=37.484842, longitude=126.930075):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m"
    # print(url)
    response = requests.get(url)
    current_data = response.json().get('current')
    # print(current_data)
    if current_data is not None:
        return current_data.get('temperature_2m')
    else:
        return'error'

In [20]:

get_weather(135,122)

'error'

In [21]:
tools = [{
    "type":"function",
    "function":{
        "name":"get_weather",
        "description":"저장된 좌표의 현재 온도를 섭씨 단위로 return합니다",
        "parameters":{
            "type":"object",
            "properties":{
                "latitude" : {"type":"number"}, 
                "longitude": {"type":"number"}
            },
            "required" : ["latitude", "longitude"], # 반드시 입력 요구 파라미터
            "additionalProperties":False # 지정된 properties외는 추가 허용 안 함
        } # parameters
    }, # function
    "strict":True # schema에 정확하게 맞는 경우만 함수 호출해라
}] # tools
messages = [{"role":"user", "content":"오늘 서울 날씨 어때요?"}]
completion = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=messages,
    # tools = tools
) # completions.create

In [22]:

completion


ChatCompletion(id='chatcmpl-Bo3ZIMZY7ZQo8td8qZsaAxa50H0WY', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='죄송하지만, 현재 실시간 날씨 정보를 제공할 수 없습니다. 서울의 오늘 날씨를 확인하시려면 주간 기상 웹사이트나 날씨 앱을 참고하시길 추천드립니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1751269816, model='gpt-4.1-nano-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_f12167b370', usage=CompletionUsage(completion_tokens=42, prompt_tokens=15, total_tokens=57, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [23]:
completion = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=messages,
    tools = tools
) # completions.create

In [24]:
completion.choices[0].message.tool_calls

[ChatCompletionMessageToolCall(id='call_sylIUQfcNl4vpRKBQu8JWzx3', function=Function(arguments='{"latitude":37.5665,"longitude":126.978}', name='get_weather'), type='function')]

In [29]:

import json
for tool_call in completion.choices[0].message.tool_calls:
    fun_name = tool_call.function.name
    args     = json.loads(tool_call.function.arguments)
    if fun_name == 'get_weather':
        result = get_weather(args['latitude'], args['longitude'])
        print('기온 :', result)

기온 : 26.5


In [32]:
completion.choices[0].message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_sylIUQfcNl4vpRKBQu8JWzx3', function=Function(arguments='{"latitude":37.5665,"longitude":126.978}', name='get_weather'), type='function')])

In [33]:
messages = [{"role":"user", "content":"오늘 서울 날씨 어때요?"}]
messages.append(completion.choices[0].message)
messages.append({
    "role":"tool",
    "tool_call_id":tool_call.id,
    "content":str(result)
})
print(messages)

[{'role': 'user', 'content': '오늘 서울 날씨 어때요?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_sylIUQfcNl4vpRKBQu8JWzx3', function=Function(arguments='{"latitude":37.5665,"longitude":126.978}', name='get_weather'), type='function')]), {'role': 'tool', 'tool_call_id': 'call_sylIUQfcNl4vpRKBQu8JWzx3', 'content': '26.5'}]


In [34]:
from pprint import pprint
pprint(messages)

[{'content': '오늘 서울 날씨 어때요?', 'role': 'user'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_sylIUQfcNl4vpRKBQu8JWzx3', function=Function(arguments='{"latitude":37.5665,"longitude":126.978}', name='get_weather'), type='function')]),
 {'content': '26.5',
  'role': 'tool',
  'tool_call_id': 'call_sylIUQfcNl4vpRKBQu8JWzx3'}]


In [35]:
completion2 = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=messages,
    tools=tools
)

In [36]:

completion2.choices[0].message.content

'오늘 서울의 기온은 약 26.5도입니다. 날씨가 따뜻하니 좋은 하루 보내세요!'